# Equivariance check

In [1]:
import json
import math
import time

import pandas as pd
import torch
from torch.utils.data import ConcatDataset, DataLoader

from dataset.pac_matrix import PAC5MatrixDataset
from gcnn.base_model import GroupEquivariantCNN
from gcnn.transform import precompute_d6_indices
from pac import smiles_to_pac

FOLD = 1
SPLIT_DIR = "../dataset/PAC5-1"
CHECKPOINT = f"../models/gcnn_1.1M-pac_grid-[PAC5-1]-fold_{FOLD}/best.ckpt"

HIDDEN_CHANNELS = 32
KERNEL_SIZE = 5
NUM_HIDDEN = 3

# Double-precision runs use first DOUBLE_SUBSET structures only
BATCH_SIZE = 256
DOUBLE_SUBSET = 4096
PAC_CHECK_MOLECULES = 25

PROPERTIES = [
    "xtb_homo",
    "xtb_lumo",
    "xtb_hl_gap",
    "pm7_homo",
    "pm7_lumo",
    "pm7_hl_gap",
    "pm7_hof",
]
PROPERTY_TYPES = ["intensive"] * 6 + ["extensive"]

# Disable TF32 to run float32 check at full precision
torch.set_float32_matmul_precision("highest")
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## Group action

In [2]:
def build_group_permutations(device):
    """Build the three permutations that take a D6 element through the network.

    Returns
    -------
    group_axis : torch.Tensor of shape (12, 12)
        `group_axis[k, h]` is the index of ``g_k^-1 g_h``, which is how the group
        axis of a feature map is relabelled. Matches `D6GroupKernel.perm12`.
    direction : torch.Tensor of shape (12, 6)
        How the six vertices of a hexagon are relabelled. Matches
        `VectorLiftingKernel.rotation_indices`.
    channel : torch.Tensor of shape (12, 13)
        How the 13 input channels are relabelled. Channel 0 is the framework and
        does not move; channels 1 to 6 and 7 to 12 follow `direction`.
    """
    group_axis = torch.zeros(12, 12, dtype=torch.long)
    for left in range(12):
        for right in range(12):
            if left < 6:
                group_axis[left, right] = (
                    (right - left) % 6 if right < 6 else 6 + (right - 6 - left) % 6
                )
            else:
                mirror = left - 6
                group_axis[left, right] = (
                    6 + (mirror - right) % 6
                    if right < 6
                    else (mirror - (right - 6)) % 6
                )

    direction = torch.zeros(12, 6, dtype=torch.long)
    for element in range(12):
        for slot in range(6):
            direction[element, slot] = (
                (slot + element) % 6 if element < 6 else (-slot - (element - 6)) % 6
            )

    zero = torch.zeros(1, dtype=torch.long)
    channel = torch.stack(
        [
            torch.cat([zero, 1 + direction[element], 7 + direction[element]])
            for element in range(12)
        ]
    )
    return group_axis.to(device), direction.to(device), channel.to(device)


def transform_input(grid, element, spatial, channel, grid_size):
    """Apply a D6 element to a batch of input grids."""
    batch_size, num_channels = grid.shape[0], grid.shape[1]
    moved = (
        grid.reshape(batch_size, num_channels, grid_size * grid_size)
        .index_select(2, spatial[element])
        .reshape(batch_size, num_channels, grid_size, grid_size)
    )
    return moved.index_select(1, channel[element])


def transform_features(features, element, spatial, group_axis, grid_size):
    """Apply a D6 element to a batch of feature maps that have a group axis."""
    batch_size, num_channels = features.shape[0], features.shape[1]
    group_size = features.shape[2]
    moved = (
        features.reshape(batch_size, num_channels, group_size, grid_size * grid_size)
        .index_select(3, spatial[element])
        .reshape(batch_size, num_channels, group_size, grid_size, grid_size)
    )
    return moved.index_select(2, group_axis[element])


GROUP_LABELS = {index: (f"r^{index}" if index else "e") for index in range(6)}
GROUP_LABELS.update(
    {6 + index: (f"r^{index}s" if index else "s") for index in range(6)}
)

## Data and model

In [3]:
data = ConcatDataset(
    [
        PAC5MatrixDataset(
            f"{SPLIT_DIR}/fold_{FOLD}_{split}.csv",
            properties=PROPERTIES,
            num_workers=12,
        )
        for split in ("train", "val", "test")
    ]
)
loader = DataLoader(data, batch_size=BATCH_SIZE, shuffle=False)

first = next(iter(loader))
first = first[0] if isinstance(first, (list, tuple)) else first
grid_size = first.shape[-1]
num_structures = len(data)
print(f"{num_structures} structures, grid {grid_size} by {grid_size}")

spatial = precompute_d6_indices(grid_size, device=device)
group_axis, direction, channel = build_group_permutations(device)


def load_model(dtype):
    model = GroupEquivariantCNN(
        property_types=PROPERTY_TYPES,
        kernel_size=KERNEL_SIZE,
        num_hidden=NUM_HIDDEN,
        hidden_channels=HIDDEN_CHANNELS,
    )
    model.load_state_dict(torch.load(CHECKPOINT, map_location="cpu"))
    return model.to(device=device, dtype=dtype).eval()


def attach_hooks(model):
    """Hook every hidden layer and the input of each pooled head."""
    features, pooled = {}, {}

    def feature_hook(name):
        def hook(module, inputs, output):
            features[name] = output.detach().clone()  # SiLU runs in place

        return hook

    def pooled_hook(name):
        def hook(module, inputs):
            pooled[name] = inputs[0].detach().clone()

        return hook

    handles = [
        model.lifting_conv.register_forward_hook(feature_hook("lift")),
        model.lift_bn.register_forward_hook(feature_hook("lift_bn")),
        model.electronic_head.register_forward_pre_hook(pooled_hook("pooled_max_mean")),
        model.thermodynamic_head.register_forward_pre_hook(
            pooled_hook("pooled_sum_mean")
        ),
    ]
    for index, (convolution, norm) in enumerate(zip(model.gconvs, model.gbns)):
        handles.append(convolution.register_forward_hook(feature_hook(f"gconv{index}")))
        handles.append(norm.register_forward_hook(feature_hook(f"gbn{index}")))
    return features, pooled, handles


probe = load_model(torch.float32)
probe_features, probe_pooled, probe_handles = attach_hooks(probe)
with torch.no_grad():
    probe(first.float().to(device))
num_hidden_layers = len(probe_features)
hidden_layer_names = list(probe_features)
print(f"{num_hidden_layers} hidden layers hooked: {hidden_layer_names}")
print("Pooled inputs hooked:", list(probe_pooled))
for handle in probe_handles:
    handle.remove()
del probe

678725 structures, grid 15 by 15
8 hidden layers hooked: ['lift', 'lift_bn', 'gconv0', 'gbn0', 'gconv1', 'gbn1', 'gconv2', 'gbn2']
Pooled inputs hooked: ['pooled_max_mean', 'pooled_sum_mean']


## Group action check

In [4]:
sample = first[: min(32, first.shape[0])].float().to(device)

identity_is_trivial = bool(
    torch.equal(transform_input(sample, 0, spatial, channel, grid_size), sample)
)

input_failures = []
for left in range(12):
    once = transform_input(sample, left, spatial, channel, grid_size)
    for right in range(12):
        twice = transform_input(once, right, spatial, channel, grid_size)
        matches = [
            element
            for element in range(12)
            if torch.equal(
                twice, transform_input(sample, element, spatial, channel, grid_size)
            )
        ]
        if len(matches) != 1:
            input_failures.append((left, right, matches))

feature_sample = torch.randn(
    4, 8, 12, grid_size, grid_size, device=device, dtype=torch.float64
)
feature_failures = []
for left in range(12):
    once = transform_features(feature_sample, left, spatial, group_axis, grid_size)
    for right in range(12):
        twice = transform_features(once, right, spatial, group_axis, grid_size)
        matches = [
            element
            for element in range(12)
            if torch.equal(
                twice,
                transform_features(
                    feature_sample, element, spatial, group_axis, grid_size
                ),
            )
        ]
        if len(matches) != 1:
            feature_failures.append((left, right, matches))

print("Identity is trivial:", identity_is_trivial)
print("Input action closes:", not input_failures, input_failures[:3])
print("Feature action closes:", not feature_failures, feature_failures[:3])

Identity is trivial: True
Input action closes: True []
Feature action closes: True []


In [5]:
frame = pd.read_csv(f"{SPLIT_DIR}/fold_{FOLD}_test.csv")
check_smiles = frame["smiles"].tolist()[:PAC_CHECK_MOLECULES]

images_matched, images_unmatched, elements_reached = 0, 0, set()
for smiles in check_smiles:
    try:
        pac = smiles_to_pac(smiles)
    except Exception:
        continue
    if pac is None:
        continue
    base = pac.to_tensor().unsqueeze(0).float().to(device)

    images = []
    rotated = pac
    for _ in range(6):
        images.append(rotated.to_tensor())
        rotated = rotated.rotated()
    reflected = pac.reflected()
    for _ in range(6):
        images.append(reflected.to_tensor())
        reflected = reflected.rotated()

    for image in images:
        target = image.unsqueeze(0).float().to(device)
        found = [
            element
            for element in range(12)
            if torch.equal(
                transform_input(base, element, spatial, channel, grid_size), target
            )
        ]
        if len(found) == 1:
            images_matched += 1
            elements_reached.add(found[0])
        else:
            images_unmatched += 1

print(f"Images matched: {images_matched}")
print(f"Images unmatched: {images_unmatched}")
print(f"Distinct elements reached: {sorted(elements_reached)}")

Images matched: 132
Images unmatched: 168
Distinct elements reached: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


## Measurement

In [6]:
class Accumulator:
    """Collect deviation statistics in float64 without storing every value."""

    def __init__(self):
        self.count = 0
        self.sum_absolute = 0.0
        self.sum_squared = 0.0
        self.max_absolute = 0.0
        self.max_reference = 0.0
        self.sum_reference_squared = 0.0

    def add(self, deviation, reference):
        deviation = deviation.abs().double()
        reference = reference.abs().double()
        self.count += deviation.numel()
        self.sum_absolute += deviation.sum().item()
        self.sum_squared += (deviation**2).sum().item()
        self.max_absolute = max(self.max_absolute, deviation.max().item())
        self.max_reference = max(self.max_reference, reference.max().item())
        self.sum_reference_squared += (reference**2).sum().item()

    def merge(self, other):
        self.count += other.count
        self.sum_absolute += other.sum_absolute
        self.sum_squared += other.sum_squared
        self.max_absolute = max(self.max_absolute, other.max_absolute)
        self.max_reference = max(self.max_reference, other.max_reference)
        self.sum_reference_squared += other.sum_reference_squared

    def summary(self):
        if self.count == 0:
            return {}
        root_mean_square = math.sqrt(self.sum_squared / self.count)
        reference_root_mean_square = math.sqrt(self.sum_reference_squared / self.count)
        return {
            "count": self.count,
            "max_absolute": self.max_absolute,
            "mean_absolute": self.sum_absolute / self.count,
            "root_mean_square": root_mean_square,
            "max_reference": self.max_reference,
            "reference_root_mean_square": reference_root_mean_square,
            "max_relative": (
                self.max_absolute / self.max_reference
                if self.max_reference > 0
                else float("nan")
            ),
            "root_mean_square_relative": (
                root_mean_square / reference_root_mean_square
                if reference_root_mean_square > 0
                else float("nan")
            ),
        }


def collapse(accumulators, feature_type):
    """Merge every layer and every element of a feature type into a single summary."""
    merged = Accumulator()
    for key, accumulator in accumulators.items():
        if key[0] == feature_type:
            merged.merge(accumulator)
    return merged.summary()


def batches(limit=None):
    """Yield input batches, stopping after `limit` structures if given."""
    seen = 0
    for batch in loader:
        grid = batch[0] if isinstance(batch, (list, tuple)) else batch
        if limit is not None and seen + grid.shape[0] > limit:
            grid = grid[: limit - seen]
        yield grid
        seen += grid.shape[0]
        if limit is not None and seen >= limit:
            return


def measure_equivariance(dtype, autocast_dtype, limit=None, label=""):
    """Deviation between the transformed feature maps and the feature maps of
    the transformed input, over every group element and every hidden layer."""
    model = load_model(torch.float32 if autocast_dtype else dtype)
    features, pooled, handles = attach_hooks(model)
    accumulators = {}
    started = time.time()

    def forward(batch):
        if autocast_dtype is None:
            model(batch)
        else:
            with torch.autocast(device_type=device.type, dtype=autocast_dtype):
                model(batch)

    def get(kind, layer, element):
        key = (kind, layer, element)
        if key not in accumulators:
            accumulators[key] = Accumulator()
        return accumulators[key]

    seen = 0
    with torch.no_grad():
        for grid in batches(limit):
            batch = grid.to(device=device, dtype=dtype)
            forward(batch)
            reference_features = {
                name: value.clone() for name, value in features.items()
            }
            reference_pooled = {name: value.clone() for name, value in pooled.items()}

            for element in range(1, 12):
                forward(transform_input(batch, element, spatial, channel, grid_size))
                for name, reference in reference_features.items():
                    expected = transform_features(
                        reference.double(), element, spatial, group_axis, grid_size
                    )
                    get("hidden", name, element).add(
                        features[name].double() - expected, expected
                    )
                for name, reference in reference_pooled.items():
                    expected = reference.double()
                    get("pooled", name, element).add(
                        pooled[name].double() - expected, expected
                    )

            seen += batch.shape[0]
            if seen % (BATCH_SIZE * 20) == 0:
                hidden = collapse(accumulators, "hidden")
                print(
                    f" + {label} {seen} structures, hidden max "
                    f"{hidden['max_absolute']:.3e}, {time.time() - started:.0f}s",
                    flush=True,
                )

    for handle in handles:
        handle.remove()
    return accumulators, seen


def measure_noise_floor(dtype, autocast_dtype, limit, label=""):
    """Rounding error of the same forward pass against a float64 reference, with no symmetry applied."""
    reference_model = load_model(torch.float64)
    reference_features, reference_pooled, reference_handles = attach_hooks(
        reference_model
    )
    model = load_model(torch.float32 if autocast_dtype else dtype)
    features, pooled, handles = attach_hooks(model)

    accumulators = {}
    with torch.no_grad():
        for grid in batches(limit):
            batch = grid.to(device)
            reference_model(batch.double())
            captured_features = {
                name: value.clone() for name, value in reference_features.items()
            }
            captured_pooled = {
                name: value.clone() for name, value in reference_pooled.items()
            }

            if autocast_dtype is None:
                model(batch.to(dtype))
            else:
                with torch.autocast(device_type=device.type, dtype=autocast_dtype):
                    model(batch.to(torch.float32))

            for name, reference in captured_features.items():
                accumulators.setdefault(("hidden", name), Accumulator()).add(
                    features[name].double() - reference, reference
                )
            for name, reference in captured_pooled.items():
                accumulators.setdefault(("pooled", name), Accumulator()).add(
                    pooled[name].double() - reference, reference
                )

    for handle in reference_handles + handles:
        handle.remove()
    return accumulators

### Double precision on subset

In [7]:
double_equivariance, double_seen = measure_equivariance(
    torch.float64, None, limit=DOUBLE_SUBSET, label="Float64"
)
double_hidden = collapse(double_equivariance, "hidden")
double_pooled = collapse(double_equivariance, "pooled")
print(f"Float64 on {double_seen} structures")
print(
    f" + Hidden max abs {double_hidden['max_absolute']:.3e}\t"
    f"relative {double_hidden['max_relative']:.3e}"
)
print(
    f" + Pooled max abs {double_pooled['max_absolute']:.3e}\t"
    f"relative {double_pooled['max_relative']:.3e}"
)

Float64 on 4096 structures
 + Hidden max abs 9.095e-13	relative 4.079e-15
 + Pooled max abs 2.274e-13	relative 4.880e-16


### Noise floor on same subset

In [8]:
float32_noise = measure_noise_floor(
    torch.float32, None, limit=DOUBLE_SUBSET, label="Float32"
)
float32_noise_hidden = collapse(float32_noise, "hidden")
float32_noise_pooled = collapse(float32_noise, "pooled")
print("Float32 against float64 on untransformed inputs")
print(
    f" + Hidden max abs {float32_noise_hidden['max_absolute']:.3e}\t"
    f"relative {float32_noise_hidden['max_relative']:.3e}"
)
print(
    f" + Pooled max abs {float32_noise_pooled['max_absolute']:.3e}\t"
    f"relative {float32_noise_pooled['max_relative']:.3e}"
)

bfloat16_noise = measure_noise_floor(
    torch.float32, torch.bfloat16, limit=DOUBLE_SUBSET, label="bfloat16"
)
bfloat16_noise_hidden = collapse(bfloat16_noise, "hidden")
bfloat16_noise_pooled = collapse(bfloat16_noise, "pooled")
print("bfloat16 autocast against float64")
print(
    f" + Hidden max abs {bfloat16_noise_hidden['max_absolute']:.3e}\t"
    f"relative {bfloat16_noise_hidden['max_relative']:.3e}"
)
print(
    f" + Pooled max abs {bfloat16_noise_pooled['max_absolute']:.3e}\t"
    f"relative {bfloat16_noise_pooled['max_relative']:.3e}"
)

Float32 against float64 on untransformed inputs
 + Hidden max abs 9.626e-04	relative 4.317e-06
 + Pooled max abs 2.495e-03	relative 5.355e-06
bfloat16 autocast against float64
 + Hidden max abs 9.603e-01	relative 4.307e-03
 + Pooled max abs 6.763e+00	relative 1.452e-02


### Single precision on all structures

In [9]:
float32_equivariance, float32_seen = measure_equivariance(
    torch.float32, None, limit=None, label="Float32"
)
float32_hidden = collapse(float32_equivariance, "hidden")
float32_pooled = collapse(float32_equivariance, "pooled")
print(f"Float32 on {float32_seen} structures")
print(
    f" + Hidden max abs {float32_hidden['max_absolute']:.3e}\t"
    f"relative {float32_hidden['max_relative']:.3e}\t"
    f"magnitude {float32_hidden['max_reference']:.3e}"
)
print(
    f" + Pooled max abs {float32_pooled['max_absolute']:.3e}\t"
    f"relative {float32_pooled['max_relative']:.3e}\t"
    f"magnitude {float32_pooled['max_reference']:.3e}"
)

 + Float32 5120 structures, hidden max 5.798e-04, 30s
 + Float32 10240 structures, hidden max 5.798e-04, 59s
 + Float32 15360 structures, hidden max 6.561e-04, 87s
 + Float32 20480 structures, hidden max 6.561e-04, 114s
 + Float32 25600 structures, hidden max 6.561e-04, 142s
 + Float32 30720 structures, hidden max 6.561e-04, 172s
 + Float32 35840 structures, hidden max 6.561e-04, 200s
 + Float32 40960 structures, hidden max 6.561e-04, 227s
 + Float32 46080 structures, hidden max 6.561e-04, 255s
 + Float32 51200 structures, hidden max 6.561e-04, 282s
 + Float32 56320 structures, hidden max 6.561e-04, 309s
 + Float32 61440 structures, hidden max 6.561e-04, 337s
 + Float32 66560 structures, hidden max 6.561e-04, 365s
 + Float32 71680 structures, hidden max 6.561e-04, 392s
 + Float32 76800 structures, hidden max 6.561e-04, 420s
 + Float32 81920 structures, hidden max 6.561e-04, 447s
 + Float32 87040 structures, hidden max 6.561e-04, 475s
 + Float32 92160 structures, hidden max 6.561e-04, 5

### bfloat16 autocast on all structures

In [10]:
bfloat16_equivariance, bfloat16_seen = measure_equivariance(
    torch.float32, torch.bfloat16, limit=None, label="bfloat16"
)
bfloat16_hidden = collapse(bfloat16_equivariance, "hidden")
bfloat16_pooled = collapse(bfloat16_equivariance, "pooled")
print(f"bfloat16 autocast on {bfloat16_seen} structures")
print(
    f" + Hidden max abs {bfloat16_hidden['max_absolute']:.3e}\t"
    f"relative {bfloat16_hidden['max_relative']:.3e}"
)
print(
    f" + Pooled max abs {bfloat16_pooled['max_absolute']:.3e}\t"
    f"relative {bfloat16_pooled['max_relative']:.3e}"
)

 + bfloat16 5120 structures, hidden max 1.000e+00, 18s
 + bfloat16 10240 structures, hidden max 1.000e+00, 36s
 + bfloat16 15360 structures, hidden max 1.000e+00, 54s
 + bfloat16 20480 structures, hidden max 1.000e+00, 72s
 + bfloat16 25600 structures, hidden max 1.000e+00, 90s
 + bfloat16 30720 structures, hidden max 1.000e+00, 108s
 + bfloat16 35840 structures, hidden max 1.000e+00, 126s
 + bfloat16 40960 structures, hidden max 1.000e+00, 143s
 + bfloat16 46080 structures, hidden max 1.000e+00, 161s
 + bfloat16 51200 structures, hidden max 1.000e+00, 179s
 + bfloat16 56320 structures, hidden max 1.000e+00, 197s
 + bfloat16 61440 structures, hidden max 1.000e+00, 214s
 + bfloat16 66560 structures, hidden max 1.000e+00, 232s
 + bfloat16 71680 structures, hidden max 1.000e+00, 250s
 + bfloat16 76800 structures, hidden max 1.000e+00, 268s
 + bfloat16 81920 structures, hidden max 1.000e+00, 286s
 + bfloat16 87040 structures, hidden max 1.000e+00, 303s
 + bfloat16 92160 structures, hidden 

## Values for manuscript

In [11]:
values = {
    "EQ_N_STRUCTURES": float32_seen,
    "EQ_N_LAYERS": num_hidden_layers,
    "EQ_PAC_IMAGES_MATCHED": images_matched,
    "EQ_PAC_IMAGES_UNMATCHED": images_unmatched,
    "EQ_FP64_SUBSET": double_seen,
    "EQ_FP64_HIDDEN_REL": double_hidden["max_relative"],
    "EQ_FP64_POOLED_REL": double_pooled["max_relative"],
    "EQ_FP32_HIDDEN_MAX": float32_hidden["max_absolute"],
    "EQ_FP32_HIDDEN_REL": float32_hidden["max_relative"],
    "EQ_FP32_POOLED_MAX": float32_pooled["max_absolute"],
    "EQ_FP32_POOLED_REL": float32_pooled["max_relative"],
    "EQ_SCALE_HIDDEN": float32_hidden["max_reference"],
    "EQ_SCALE_POOLED": float32_pooled["max_reference"],
    "EQ_NOISE_HIDDEN": float32_noise_hidden["max_absolute"],
    "EQ_NOISE_POOLED": float32_noise_pooled["max_absolute"],
    "EQ_BF16_HIDDEN_MAX": bfloat16_hidden["max_absolute"],
    "EQ_BF16_HIDDEN_REL": bfloat16_hidden["max_relative"],
    "EQ_BF16_POOLED_MAX": bfloat16_pooled["max_absolute"],
    "EQ_BF16_POOLED_REL": bfloat16_pooled["max_relative"],
    "EQ_BF16_NOISE_HIDDEN": bfloat16_noise_hidden["max_absolute"],
    "EQ_BF16_NOISE_POOLED": bfloat16_noise_pooled["max_absolute"],
}

print("======[ Placeholder values ]======")
for name, value in values.items():
    if isinstance(value, int):
        print(f" + {name}: {value}")
    else:
        print(f" + {name}: {value:.3e}")

with open("equivariance_numbers.json", "w") as handle:
    json.dump(values, handle, indent=1)

rows = []
for source, precision in (
    (double_equivariance, "float64"),
    (float32_equivariance, "float32"),
    (bfloat16_equivariance, "bfloat16"),
):
    for (feature_type, layer, element), accumulator in source.items():
        rows.append(
            dict(
                precision=precision,
                feature_type=feature_type,
                layer=layer,
                g=GROUP_LABELS[element],
                **accumulator.summary(),
            )
        )
table = pd.DataFrame(rows)
table.to_csv("equivariance_per_layer.csv", index=False)
print(
    f"\nWrote equivariance_numbers.json and equivariance_per_layer.csv "
    f"({len(table)} rows)"
)

======[ Placeholder values ]======
 + EQ_N_STRUCTURES: 678725
 + EQ_N_LAYERS: 8
 + EQ_PAC_IMAGES_MATCHED: 132
 + EQ_PAC_IMAGES_UNMATCHED: 168
 + EQ_FP64_SUBSET: 4096
 + EQ_FP64_HIDDEN_REL: 4.079e-15
 + EQ_FP64_POOLED_REL: 4.880e-16
 + EQ_FP32_HIDDEN_MAX: 6.714e-04
 + EQ_FP32_HIDDEN_REL: 2.146e-06
 + EQ_FP32_POOLED_MAX: 1.831e-04
 + EQ_FP32_POOLED_REL: 2.439e-07
 + EQ_SCALE_HIDDEN: 3.128e+02
 + EQ_SCALE_POOLED: 7.507e+02
 + EQ_NOISE_HIDDEN: 9.626e-04
 + EQ_NOISE_POOLED: 2.495e-03
 + EQ_BF16_HIDDEN_MAX: 2.000e+00
 + EQ_BF16_HIDDEN_REL: 6.369e-03
 + EQ_BF16_POOLED_MAX: 1.805e-01
 + EQ_BF16_POOLED_REL: 2.407e-04
 + EQ_BF16_NOISE_HIDDEN: 9.603e-01
 + EQ_BF16_NOISE_POOLED: 6.763e+00

Wrote equivariance_numbers.json and equivariance_per_layer.csv (330 rows)


In [12]:
table = pd.read_csv("equivariance_per_layer.csv")
maxima = table.groupby(["precision", "feature_type"])[
    ["max_absolute", "max_reference", "max_relative"]
].max()
print("======[ Maxima by precision and feature type ]======")
for (precision, feature_type), row in maxima.iterrows():
    print(
        f" + {precision} {feature_type}: max_absolute {row['max_absolute']:e}\t"
        f"max_reference {row['max_reference']:f}\tmax_relative {row['max_relative']:e}"
    )

======[ Maxima by precision and feature type ]======
 + bfloat16 hidden: max_absolute 2.000000e+00	max_reference 314.000000	max_relative 1.010101e-02
 + bfloat16 pooled: max_absolute 1.805267e-01	max_reference 750.060059	max_relative 5.434783e-03
 + float32 hidden: max_absolute 6.713867e-04	max_reference 312.796478	max_relative 2.593529e-06
 + float32 pooled: max_absolute 1.831055e-04	max_reference 750.668091	max_relative 6.627830e-07
 + float64 hidden: max_absolute 9.094947e-13	max_reference 222.985167	max_relative 4.675668e-15
 + float64 pooled: max_absolute 2.273737e-13	max_reference 465.941245	max_relative 9.733768e-16
